# GVS chatbot: local model test on a Colab T4

Sets up everything on a fresh Colab GPU runtime, then scores each model the same way
`docs/local-model.md` does on the desktop:

1. checks the GPU, installs Ollama and uv, starts Ollama (on `127.0.0.1` only)
2. pulls the models listed in **Settings**
3. clones the public repo
4. runs `gvs-local-check` for each model and records GPU memory
5. scores every model on the held-out and dev sets: **rules only**, **rules → model** (what visitors get), **model alone**
6. shows a table and zips the logs for download

**Before you run it**
- *Runtime → Change runtime type → T4 GPU*
- Then *Runtime → Run all*. The first run takes about 15–30 min, most of it model downloads.

Only the synthetic test messages in `evals/` are sent to the models. Don't paste real customer data into Colab.

In [ ]:
# ----------------------------------------------------------------- Settings
REPO = "AakashBhat1/gvs-chatbot"
BRANCH = "main"

# T4 has 15 GB usable VRAM, so the 8-bit 9B fits here (it does not on the 3060 Ti).
MODELS = [
    "qwen3.5:9b",         # default tag, 4-bit  (what the desktop runs)
    "qwen3.5:9b-q8_0",    # 8-bit, ~10 GB: best quality that fits a T4
    "qwen3:8b",           # previous generation, for comparison
    # "qwen3.5:4b-q8_0",  # small and fast, uncomment to compare
]
EVAL_SETS = ["evals/holdout.jsonl", "evals/understanding.jsonl"]
MODES = ["cascade", "solo"]   # cascade = rules -> model, solo = model alone
RUNS = 1                      # temperature is 0, so 1 run is enough to compare models
NUM_CTX = 8192
TIMEOUT = 60                  # seconds per message; T4 is slower than a 3060 Ti

WORK = "/content/gvs"
RESULTS = "/content/gvs-results"

In [ ]:
# ----------------------------------------------------------------- helpers + GPU check
import json, os, re, shlex, subprocess, sys, time, urllib.request
from pathlib import Path

Path(RESULTS).mkdir(parents=True, exist_ok=True)

def sh(cmd, *, env=None, check=True, quiet=False, timeout=None):
    """Run a shell command, stream its output, return (code, output)."""
    full_env = {**os.environ, **(env or {})}
    proc = subprocess.Popen(cmd, shell=True, env=full_env, text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    lines = []
    try:
        for line in proc.stdout:
            lines.append(line)
            if not quiet:
                print(line, end="")
        code = proc.wait(timeout=timeout)
    except subprocess.TimeoutExpired:
        proc.kill()
        raise
    out = "".join(lines)
    if check and code != 0:
        raise RuntimeError(f"command failed ({code}): {cmd[:120]}")
    return code, out

code, gpu = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader",
               check=False)
if code != 0:
    raise SystemExit("No GPU. Runtime -> Change runtime type -> T4 GPU, then run again.")
print("Python", sys.version.split()[0])

In [ ]:
# ----------------------------------------------------------------- install Ollama + uv
# zstd: the Ollama installer needs it; pciutils/lshw: lets it detect the NVIDIA GPU.
sh("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null", quiet=True)
if subprocess.run("command -v ollama", shell=True, capture_output=True).returncode != 0:
    sh("curl -fsSL https://ollama.com/install.sh | sh", quiet=True)
sh("ollama --version", check=False)
sh(f"{sys.executable} -m pip install -q -U uv", quiet=True)
sh("uv --version")

In [ ]:
# ----------------------------------------------------------------- start Ollama
OLLAMA = "http://127.0.0.1:11434"
ollama_env = {
    "OLLAMA_HOST": "127.0.0.1:11434",   # never 0.0.0.0: Ollama has no login
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0",
    "OLLAMA_MAX_LOADED_MODELS": "1",    # one model at a time, so memory numbers are clean
    "OLLAMA_KEEP_ALIVE": "30m",
}

def ollama_up():
    try:
        with urllib.request.urlopen(f"{OLLAMA}/api/version", timeout=2) as r:
            return json.load(r)["version"]
    except Exception:
        return None

if not ollama_up():
    log = open(f"{RESULTS}/ollama-serve.log", "a")
    subprocess.Popen(["ollama", "serve"], env={**os.environ, **ollama_env},
                     stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        if ollama_up():
            break
        time.sleep(1)
print("Ollama", ollama_up() or "DID NOT START, see ollama-serve.log")

In [ ]:
# ----------------------------------------------------------------- pull models
def pull(model):
    req = urllib.request.Request(f"{OLLAMA}/api/pull", method="POST",
                                 data=json.dumps({"model": model}).encode(),
                                 headers={"Content-Type": "application/json"})
    last, shown = None, -10
    with urllib.request.urlopen(req, timeout=3600) as resp:
        for raw in resp:
            msg = json.loads(raw)
            if "error" in msg:
                raise RuntimeError(f"{model}: {msg['error']}")
            status = msg.get("status", "")
            if msg.get("total"):
                pct = int(100 * msg.get("completed", 0) / msg["total"])
                if pct >= shown + 10:
                    print(f"  {model}: {status} {pct}%")
                    shown = pct
            elif status != last:
                print(f"  {model}: {status}")
                last = status

for m in MODELS:
    t = time.time()
    pull(m)
    print(f"{m} ready in {time.time() - t:.0f}s\n")
sh("ollama list")

In [ ]:
# ----------------------------------------------------------------- clone repo + install
if Path(WORK, ".git").exists():
    sh(f"git -C {WORK} fetch -q origin {BRANCH} && git -C {WORK} reset -q --hard origin/{BRANCH}")
else:
    sh(f"git clone -q --branch {BRANCH} https://github.com/{REPO}.git {WORK}")
sh(f"git -C {WORK} log --oneline -1")

BACKEND = f"{WORK}/backend"
sh(f"cd {BACKEND} && uv sync --locked -q")
sh(f"cd {BACKEND} && uv run -q pytest -q -p no:cacheprovider 2>&1 | tail -1")

In [ ]:
# ----------------------------------------------------------------- gvs-local-check per model
def bot_env(model, brains="rules,local"):
    return {
        "GVS_BRAINS": brains,
        "GVS_LOCAL_URL": OLLAMA,
        "GVS_LOCAL_MODEL": model,
        "GVS_LOCAL_TIMEOUT": str(TIMEOUT),
        "GVS_LOCAL_NUM_CTX": str(NUM_CTX),
    }

def gpu_used_mib():
    _, out = sh("nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits",
                quiet=True, check=False)
    return int(out.strip().splitlines()[0]) if out.strip() else None

health = {}
for m in MODELS:
    print(f"==================== {m}")
    code, out = sh(f"cd {BACKEND} && uv run -q gvs-local-check", env=bot_env(m), check=False)
    _, ps = sh("ollama ps", check=False)
    warm = re.search(r"warm run: ([\d.]+)s", out)
    health[m] = {
        "check": "OK" if code == 0 else "FAIL",
        "warm_s": float(warm.group(1)) if warm else None,
        "gpu_mib": gpu_used_mib(),
        "on_gpu": "100% GPU" in ps,
    }
    print()
health

In [ ]:
# ----------------------------------------------------------------- score every model
SCORE = re.compile(r"^\s+(\w+)\s+([\d.]+)%$", re.M)

def run_eval(label, env, extra, case_set):
    t = time.time()
    _, out = sh(f"cd {BACKEND} && uv run -q gvs-eval --set {case_set} --runs {RUNS} --show 50 {extra}",
                env=env, check=False, quiet=True)
    took = time.time() - t
    name = f"{label}__{Path(case_set).stem}".replace(":", "_").replace("/", "_")
    Path(RESULTS, f"{name}.txt").write_text(out)
    scores = {k: float(v) for k, v in SCORE.findall(out)}
    cases = int(m.group(1)) if (m := re.search(r"cases: (\d+)", out)) else 0
    return {"exact": scores.get("exact"), "travellers": scores.get("travellers"),
            "on_topic": scores.get("on_topic"), "confident": scores.get("confident"),
            "s_per_msg": round(took / max(1, cases * RUNS), 2), "minutes": round(took / 60, 1)}

rows = []
for case_set in EVAL_SETS:
    rows.append({"model": "(no AI)", "mode": "rules", "set": Path(case_set).stem,
                 **run_eval("rules", {"GVS_BRAINS": "rules"}, "--brains rules", case_set)})
    print(rows[-1])
for m in MODELS:
    if health.get(m, {}).get("check") != "OK":
        print(f"skipping {m}: gvs-local-check failed")
        continue
    for case_set in EVAL_SETS:
        for mode in MODES:
            extra = "--brains rules,local" if mode == "cascade" else "--brains local --solo"
            r = run_eval(f"{m}__{mode}", bot_env(m), extra, case_set)
            rows.append({"model": m, "mode": mode, "set": Path(case_set).stem, **r})
            print(rows[-1])

import pandas as pd
df = pd.DataFrame(rows)
df.to_csv(f"{RESULTS}/scores.csv", index=False)
Path(RESULTS, "health.json").write_text(json.dumps(health, indent=2))
df.pivot_table(index=["model", "mode"], columns="set", values=["exact", "s_per_msg"])

In [ ]:
# ----------------------------------------------------------------- download results
import shutil
from google.colab import files

sh(f"nvidia-smi > {RESULTS}/nvidia-smi.txt; ollama list > {RESULTS}/ollama-list.txt", check=False)
sh(f"cd {WORK} && git log --oneline -1 > {RESULTS}/commit.txt", check=False)
archive = shutil.make_archive("/content/gvs-results", "zip", RESULTS)
files.download(archive)

## Try it yourself

Runs the terminal bot with the model you pick. Type in the box under the cell; `/quit` to stop.
`-v` shows which brain answered each message.

In [ ]:
CHAT_MODEL = MODELS[0]
env = " ".join(f"{k}={shlex.quote(v)}" for k, v in bot_env(CHAT_MODEL).items())
!cd {BACKEND} && env {env} uv run -q gvs-chat -v